# Demo de NPCore

Este notebook muestra cómo usar **npcore**, una librería de Python para simular NPCs (personajes no jugables) con toma de decisiones probabilística.

## Objetivo

Construiremos un ejemplo simple donde distintos NPCs reaccionan según su contexto, en este caso sus puntos de vida, y observaremos cómo cambian sus decisiones a lo largo de una simulación.

## 1. Configuración inicial

Como la librería está en desarrollo local, primero agregamos la carpeta `src` al path para poder importar `npcore`.

In [1]:
!pip install npcore


ERROR: Could not find a version that satisfies the requirement npcore (from versions: none)
ERROR: No matching distribution found for npcore


In [3]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

## 2. Importar componentes principales

Importamos las tres piezas centrales de la librería:

- `Brain`: motor de decisiones
- `NPC`: personaje no jugable
- `Environment`: entorno de simulación

In [4]:
from npcore.brain import Brain
from npcore.npc import NPC
from npcore.environment import Environment


## 3. Definir la lógica de decisión

En este ejemplo, los NPCs están en estado de combate.  
La decisión dependerá de su vida (`hp`):

- Si la vida es baja, tendrán mayor probabilidad de huir
- Si la vida es media, serán más defensivos
- Si la vida es alta, tenderán a atacar

In [5]:
brain = Brain()

def combat_rule(ctx):
    hp = ctx.get("hp", 100)

    if hp < 30:
        return {"flee": 0.8, "defend": 0.2}
    elif hp < 60:
        return {"attack": 0.4, "defend": 0.6}
    else:
        return {"attack": 0.8, "defend": 0.2}

brain.add_rule("combat", combat_rule)

## 4. Crear NPCs con diferentes contextos

Ahora creamos dos NPCs:

- `Goblin`, con poca vida
- `Knight`, con mucha vida

Ambos estarán en estado de combate, pero su contexto hará que sus decisiones sean distintas.

In [6]:
npc1 = NPC("Goblin", brain)
npc2 = NPC("Knight", brain)

npc1.set_state("combat")
npc2.set_state("combat")

npc1.update_context(hp=20)
npc2.update_context(hp=80)

## 5. Crear el entorno de simulación

Agregamos ambos NPCs a un entorno para ejecutar varios pasos de simulación.

In [7]:
env = Environment()
env.add_npc(npc1)
env.add_npc(npc2)


## 6. Ejecutar la simulación

A continuación simulamos varios pasos.  
Como las decisiones son probabilísticas, los resultados pueden variar entre ejecuciones.


In [9]:
for step in range(5):
    results = env.step()
    print(f"Step {step}: {results}")

Step 0: [('Goblin', 'flee'), ('Knight', 'attack')]
Step 1: [('Goblin', 'flee'), ('Knight', 'defend')]
Step 2: [('Goblin', 'flee'), ('Knight', 'attack')]
Step 3: [('Goblin', 'flee'), ('Knight', 'attack')]
Step 4: [('Goblin', 'flee'), ('Knight', 'attack')]


## 7. Interpretación del resultado

Esperamos observar algo como:

- El `Goblin` elige con más frecuencia `flee`
- El `Knight` elige con más frecuencia `attack`

Esto muestra cómo el mismo sistema de decisión puede producir comportamientos distintos según el contexto del NPC.

## Segundo ejemplo: guardia patrullando

Ahora construiremos otro NPC con un comportamiento distinto.  
Este guardia no depende de vida, sino de si detecta una amenaza cercana.

brain_guard = Brain()

def guard_rule(ctx):
    enemy_near = ctx.get("enemy_near", False)

    if enemy_near:
        return {"attack": 0.6, "alert": 0.4}
    return {"patrol": 0.7, "rest": 0.3}

brain_guard.add_rule("guard", guard_rule)

guard = NPC("Guard", brain_guard)
guard.set_state("guard")
guard.update_context(enemy_near=True)

print("Acción del guardia:", guard.act())

In [10]:
brain_guard = Brain()

def guard_rule(ctx):
    enemy_near = ctx.get("enemy_near", False)

    if enemy_near:
        return {"attack": 0.6, "alert": 0.4}
    return {"patrol": 0.7, "rest": 0.3}

brain_guard.add_rule("guard", guard_rule)

guard = NPC("Guard", brain_guard)
guard.set_state("guard")
guard.update_context(enemy_near=True)

print("Acción del guardia:", guard.act())

Acción del guardia: attack


## Tercer ejemplo: aldeano en peligro

Este ejemplo muestra que la librería no se limita al combate.  
También puede modelar comportamientos civiles o sociales simples.

In [11]:
brain_villager = Brain()

def villager_rule(ctx):
    danger = ctx.get("danger", False)

    if danger:
        return {"hide": 0.8, "run": 0.2}
    return {"work": 0.6, "walk": 0.4}

brain_villager.add_rule("villager", villager_rule)

villager = NPC("Villager", brain_villager)
villager.set_state("villager")
villager.update_context(danger=False)

print("Acción del aldeano:", villager.act())

Acción del aldeano: walk


## Conclusión

En este notebook vimos cómo usar `npcore` para:

- definir reglas de decisión probabilística
- crear NPCs con distintos estados y contextos
- ejecutar simulaciones simples en un entorno
- reutilizar la misma librería en diferentes escenarios

Esto demuestra que `npcore` puede servir como base para prototipos de juegos, simulaciones educativas y sistemas simples de agentes.